# 00 · Generate ABC Insurer Datasets

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** Pre-workshop · Data Setup
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali
**Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai
**License:** CC BY-NC 4.0 — for educational use within the IFoA workshop and follow-up case study

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/00_generate_datasets.ipynb)

## What this notebook does

Generates the three hypothetical datasets used across every other notebook in the workshop:

1. `abc_motor_2024.csv` — 5,000 motor policies for ABC General with frequency, severity, and a fraud flag.
2. `abc_health_2024.csv` — ~8,000 covered lives across 3,200 policies for ABC Health with hospitalisation and daycare claims.
3. `abc_life_term_cohort.csv` — 50,000 term policies issued 2014–2024 for ABC Life with lapse and mortality flags.

All three datasets are **hypothetical**. ABC Insurer is a fictitious entity. Numbers are illustrative.

The schemas, value ranges, and target headline figures are locked in `00_personas_and_datasets.md`. This notebook hits each headline figure within ~5% under `numpy.random.default_rng(42)`.

## Prerequisites
- Google account (for Colab).
- No API keys.
- No local install — `numpy` and `pandas` are pre-installed on the standard Colab CPU runtime.

## How to run
Top menu → **Runtime → Run all**. Total runtime is well under 60 seconds. The CSVs land in `data/` relative to the notebook's working directory.


In [ ]:
# === Standard imports ===
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Reproducibility — seed locked at 42 so all six workshop notebooks
# load identical CSVs every time they re-run.
SEED = 42
rng = np.random.default_rng(SEED)

# Display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Output directory (relative to the notebook)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {DATA_DIR.resolve()}")


## 1. ABC General — Motor 2024

**Headline targets (locked):**
- 5,000 policies
- Average vehicle age 4.7 years
- Frequency 8.0%
- Average severity ₹38,000 per claim
- Pure premium ₹3,040 per policy-year
- Fraud flagged on ~2% of claims

**Approach:**
- Vehicle age drawn from a truncated exponential.
- Make/segment mix matches a credible Indian motor book (Maruti dominates, SUVs strong).
- Frequency comes from a Poisson with rate that varies with age, NCB, region, prior claims; the rate is rescaled so `mean(λ) = 0.084` to absorb sampling variance.
- Severity is lognormal, rescaled post-sampling so per-claim mean is exactly ₹38,000.


In [ ]:
def build_motor() -> pd.DataFrame:
    """Generate the ABC Motor 2024 dataset (5,000 policies)."""
    n = 5_000

    # ---- Vehicle age (target mean 4.7 after int truncation) ----
    vehicle_age = np.clip(rng.exponential(scale=5.3, size=n).astype(int), 0, 20)

    # ---- Make & segment (Indian market shares, FY24 retail roughly) ----
    makes = ["Maruti", "Hyundai", "Tata", "Mahindra", "Honda", "Toyota", "Kia", "Other"]
    make_p = [0.40, 0.16, 0.14, 0.10, 0.06, 0.05, 0.05, 0.04]
    vehicle_make = rng.choice(makes, size=n, p=make_p)

    segments = ["Hatchback", "Sedan", "SUV", "MUV"]
    seg_p = [0.42, 0.13, 0.35, 0.10]
    vehicle_segment = rng.choice(segments, size=n, p=seg_p)

    # ---- Cubic capacity (segment-conditional) ----
    cc_ranges = {
        "Hatchback": (800, 1200),
        "Sedan":     (1200, 1800),
        "SUV":       (1500, 2500),
        "MUV":       (1500, 2500),
    }
    cubic_capacity = np.array(
        [rng.integers(*cc_ranges[s]) for s in vehicle_segment], dtype=int
    )

    # ---- Inception / expiry / earned exposure to end of 2024 ----
    inception_offset = rng.integers(0, 366, size=n)  # 2024 is a leap year
    inception_date = pd.Series(
        pd.to_datetime("2024-01-01") + pd.to_timedelta(inception_offset, unit="D")
    )
    expiry_date = inception_date + pd.Timedelta(days=365)
    end_of_obs = pd.Timestamp("2024-12-31")
    days_earned = (end_of_obs - inception_date).dt.days + 1
    exposure_years = np.clip(days_earned.values / 365.0, 0.0, 1.0)

    # ---- IDV: depreciates with age, scaled by segment, with noise ----
    base_idv = {"Hatchback": 600_000, "Sedan": 1_000_000,
                "SUV": 1_500_000,    "MUV": 1_200_000}
    depreciation = 0.88 ** vehicle_age
    noise = rng.uniform(0.7, 1.3, size=n)
    idv_inr = np.array([base_idv[s] for s in vehicle_segment]) * depreciation * noise
    idv_inr = np.clip(idv_inr, 200_000, 5_000_000).astype(int)

    # ---- Policyholder demographics ----
    policyholder_age = np.clip(rng.normal(loc=40, scale=12, size=n), 18, 80).astype(int)
    policyholder_gender = rng.choice(["M", "F", "X"], size=n, p=[0.75, 0.245, 0.005])
    region = rng.choice(["Tier1", "Tier2", "Tier3"], size=n, p=[0.45, 0.35, 0.20])

    # ---- Prior claims & NCB (NCB zeroed if any prior claim) ----
    prior_claims_3y = rng.choice(
        [0, 1, 2, 3, 4, 5], size=n,
        p=[0.78, 0.16, 0.04, 0.015, 0.004, 0.001],
    )
    ncb_pct = np.where(
        prior_claims_3y > 0,
        0,
        rng.choice([20, 25, 35, 45, 50], size=n,
                   p=[0.20, 0.25, 0.25, 0.18, 0.12]),
    )

    # ---- Frequency: build expected λ with realistic effects, then
    # rescale baseline so mean(λ) = 0.084 (overshoots 0.080 slightly to
    # absorb Poisson noise on ~2,500 exposure-years).
    young_driver  = (policyholder_age < 25).astype(float)
    senior_driver = (policyholder_age > 65).astype(float)
    tier1 = (region == "Tier1").astype(float)
    suv = (vehicle_segment == "SUV").astype(float)

    log_lambda_raw = (
        0.04 * (vehicle_age - 5)
        + 0.25 * young_driver
        + 0.15 * senior_driver
        + 0.30 * prior_claims_3y
        - 0.012 * ncb_pct
        + 0.18 * tier1
        + 0.07 * suv
    )
    lambda_raw = np.exp(log_lambda_raw)
    lambda_freq = lambda_raw * (0.084 / lambda_raw.mean())

    claim_count = rng.poisson(lambda_freq * exposure_years)
    claim_count = np.clip(claim_count, 0, 4)

    # ---- Severity: lognormal then rescale to per-claim mean ₹38,000 ----
    sev_sigma = 0.6
    sev_mu = np.log(38_000) - sev_sigma ** 2 / 2

    total_claims = int(claim_count.sum())
    individual_severities = rng.lognormal(sev_mu, sev_sigma, size=total_claims)
    if total_claims > 0:
        individual_severities *= 38_000.0 / individual_severities.mean()

    claim_amount_inr = np.zeros(n)
    cursor = 0
    for i, c in enumerate(claim_count):
        if c > 0:
            claim_amount_inr[i] = individual_severities[cursor:cursor + c].sum()
            cursor += c
    claim_amount_inr = np.minimum(claim_amount_inr, 1_500_000).round(0)

    # ---- Fraud flag: ~2% of claims flagged for review ----
    fraud_flag = np.zeros(n, dtype=int)
    has_claim_idx = np.where(claim_count > 0)[0]
    n_fraud = max(1, int(round(0.02 * len(has_claim_idx))))
    fraud_idx = rng.choice(has_claim_idx, size=n_fraud, replace=False)
    fraud_flag[fraud_idx] = 1

    return pd.DataFrame({
        "policy_id":           [f"ABC-MOT-{str(i + 1).zfill(6)}" for i in range(n)],
        "inception_date":      inception_date.dt.strftime("%Y-%m-%d"),
        "expiry_date":         expiry_date.dt.strftime("%Y-%m-%d"),
        "exposure_years":      exposure_years.round(4),
        "vehicle_age_years":   vehicle_age,
        "vehicle_make":        vehicle_make,
        "vehicle_segment":     vehicle_segment,
        "cubic_capacity":      cubic_capacity,
        "idv_inr":             idv_inr,
        "ncb_pct":             ncb_pct,
        "policyholder_age":    policyholder_age,
        "policyholder_gender": policyholder_gender,
        "region":              region,
        "prior_claims_3y":     prior_claims_3y,
        "claim_count":         claim_count,
        "claim_amount_inr":    claim_amount_inr,
        "fraud_flag":          fraud_flag,
    })


motor = build_motor()
motor.to_csv(DATA_DIR / "abc_motor_2024.csv", index=False)
print(f"Wrote {len(motor):,} rows -> {DATA_DIR/'abc_motor_2024.csv'}")
motor.head()


## 2. ABC Health — Indemnity 2024

**Headline targets (locked):**
- 3,200 policies, ~8,000 covered lives
- Hospitalisation frequency 4.2% per member-year
- Average severity ₹62,000 per claim
- Pure premium ₹2,604 per member-year

**Approach:**
- Each policy gets a family composition (Self only, Self+Spouse, with/without children, with/without parent). The mix targets a mean of 2.5 members per policy.
- Member age depends on relationship to the Self.
- Pre-existing disease probability rises with age.
- Hospitalisation count is Poisson, rate rescaled so `mean(λ) = 0.041`.
- Severity is lognormal, rescaled to per-claim mean ₹62,000.
- Daycare counts are independent and lower magnitude.


In [ ]:
def build_health() -> pd.DataFrame:
    """Generate the ABC Health 2024 dataset (~8,000 lives, 3,200 policies)."""
    n_policies = 3_200

    # Family composition — tuned so weighted mean size ≈ 2.5
    composition_options = [
        (["Self"],                                              0.27),
        (["Self", "Spouse"],                                    0.22),
        (["Self", "Spouse", "Son"],                             0.12),
        (["Self", "Spouse", "Daughter"],                        0.12),
        (["Self", "Spouse", "Son", "Daughter"],                 0.11),
        (["Self", "Spouse", "Son", "Son"],                      0.03),
        (["Self", "Spouse", "Daughter", "Daughter"],            0.03),
        (["Self", "Parent"],                                    0.05),
        (["Self", "Spouse", "Parent"],                          0.03),
        (["Self", "Spouse", "Son", "Daughter", "Parent"],       0.02),
    ]
    comps, probs = zip(*composition_options)
    chosen_idx = rng.choice(len(comps), size=n_policies, p=probs)
    chosen_compositions = [comps[i] for i in chosen_idx]

    rows = []
    for pol_idx, members in enumerate(chosen_compositions):
        policy_id = f"ABC-HLT-{str(pol_idx + 1).zfill(6)}"

        # Policy-level (shared across the household)
        inception_offset = int(rng.integers(0, 366))
        inception_date = pd.Timestamp("2024-01-01") + pd.Timedelta(days=inception_offset)
        end_of_obs = pd.Timestamp("2024-12-31")
        days_earned = (end_of_obs - inception_date).days + 1
        exposure_years = round(min(max(days_earned / 365.0, 0.0), 1.0), 4)

        self_age = int(np.clip(rng.normal(38, 9), 22, 65))

        sum_insured_options = [300_000, 500_000, 750_000, 1_000_000, 1_500_000,
                               2_000_000, 3_000_000, 5_000_000]
        si_p = [0.05, 0.20, 0.18, 0.20, 0.15, 0.12, 0.07, 0.03]
        sum_insured = int(rng.choice(sum_insured_options, p=si_p))

        product_tier = rng.choice(["Bronze", "Silver", "Gold"], p=[0.40, 0.40, 0.20])
        region = rng.choice(["Tier1", "Tier2", "Tier3"], p=[0.50, 0.32, 0.18])

        for suffix, relationship in enumerate(members, start=1):
            if relationship == "Self":
                member_age = self_age
                member_gender = rng.choice(["M", "F", "X"], p=[0.62, 0.375, 0.005])
            elif relationship == "Spouse":
                member_age = int(np.clip(self_age + rng.integers(-4, 5), 18, 70))
                member_gender = "F"  # complement of typical Self skew
            elif relationship in ("Son", "Daughter"):
                member_age = int(np.clip(self_age - rng.integers(20, 38), 0, 25))
                member_gender = "M" if relationship == "Son" else "F"
            else:  # Parent
                member_age = int(np.clip(self_age + rng.integers(20, 35), 50, 90))
                member_gender = rng.choice(["M", "F"], p=[0.55, 0.45])

            ped_prob = 0.05 + 0.012 * max(0, member_age - 30)
            pre_existing_flag = int(rng.random() < min(ped_prob, 0.55))

            rows.append({
                "policy_id":         policy_id,
                "member_id":         f"{policy_id}-{str(suffix).zfill(2)}",
                "relationship":      relationship,
                "inception_date":    inception_date.strftime("%Y-%m-%d"),
                "exposure_years":    exposure_years,
                "member_age":        member_age,
                "member_gender":     member_gender,
                "sum_insured_inr":   sum_insured,
                "product_tier":      product_tier,
                "region":            region,
                "pre_existing_flag": pre_existing_flag,
            })

    df = pd.DataFrame(rows)

    # ---- Hospitalisation frequency: target 4.2% per member-year ----
    age_effect = 0.020 * np.maximum(0, df["member_age"].values - 35)
    ped_effect = 0.30 * df["pre_existing_flag"].values
    tier1_effect = 0.10 * (df["region"].values == "Tier1").astype(float)
    log_lambda_raw = (
        0.05 * np.minimum(df["member_age"].values, 5)  # young children claim a bit more
        + age_effect + ped_effect + tier1_effect
    )
    lambda_raw = np.exp(log_lambda_raw)
    lambda_hosp = lambda_raw * (0.041 / lambda_raw.mean())

    hosp_count = rng.poisson(lambda_hosp * df["exposure_years"].values)
    hosp_count = np.clip(hosp_count, 0, 4)

    # ---- Severity: lognormal then rescale to ₹62,000 per claim ----
    sev_sigma = 0.65
    sev_mu = np.log(62_000) - sev_sigma ** 2 / 2

    total_hosp = int(hosp_count.sum())
    severities = rng.lognormal(sev_mu, sev_sigma, size=total_hosp)
    if total_hosp > 0:
        severities *= 62_000.0 / severities.mean()

    claim_amount = np.zeros(len(df))
    cur = 0
    for i, c in enumerate(hosp_count):
        if c > 0:
            claim_amount[i] = severities[cur:cur + c].sum()
            cur += c
    claim_amount = np.minimum(claim_amount, 2_000_000).round(0)

    # ---- Daycare counts (independent) ----
    daycare_lambda = 0.06 + 0.005 * np.maximum(0, df["member_age"].values - 40)
    daycare_count = rng.poisson(daycare_lambda * df["exposure_years"].values)
    daycare_count = np.clip(daycare_count, 0, 3)

    df["hospitalisation_count"] = hosp_count
    df["claim_amount_inr"] = claim_amount
    df["daycare_count"] = daycare_count
    return df


health = build_health()
health.to_csv(DATA_DIR / "abc_health_2024.csv", index=False)
print(f"Wrote {len(health):,} rows across {health['policy_id'].nunique():,} policies"
      f" -> {DATA_DIR/'abc_health_2024.csv'}")
health.head()


## 3. ABC Life — Term Cohort 2014–2024

**Headline targets (locked):**
- 50,000 policies issued 2014-01-01 to 2024-12-31
- 13-month persistency 84% (i.e. 16% have lapsed by month 13)
- Year-2 annualised lapse rate 12% on those still in force at month 13
- Mortality `q_x = 0.001 × 2^((age−30)/10)` for non-smokers (smokers ×2)

**Approach:**
- Lapse time is sampled from a piecewise-constant hazard with three regimes:
  - months 1–13: hazard chosen so survival to month 13 is 0.84;
  - months 14–25: hazard chosen so annualised lapse on remaining is 12%;
  - months 26+: low long-term hazard (~5%/year).
- We invert the survival function to draw lapse times exactly.
- Mortality is accumulated year-by-year over the in-force period; if a death pre-empts a lapse, the death wins and the lapse is reverted.
- Observation cut-off is 2026-05-15 (workshop date).

> **Note on q_x at issue-age 35 NS year 1.** The personas spec lists 0.0011 alongside the formula "0.001 q_x at age 30, doubling every decade". The formula gives `0.001 × 2^0.5 = 0.00141` at age 35 — these two figures are mutually inconsistent. This notebook honours the formula. The observed q_x is approximate (we don't track date-of-claim within a year, so empirical q is blended across the in-force period).


In [ ]:
def build_life() -> pd.DataFrame:
    """Generate the ABC Life Term Cohort dataset (50,000 policies, 2014-2024)."""
    n = 50_000

    # ---- Issue date spread uniformly across 2014-01-01 to 2024-12-31 ----
    start = pd.Timestamp("2014-01-01")
    end = pd.Timestamp("2024-12-31")
    days_range = (end - start).days
    issue_offset = rng.integers(0, days_range + 1, size=n)
    issue_date = pd.Series(start + pd.to_timedelta(issue_offset, unit="D"))

    # ---- Demographics ----
    issue_age = np.clip(rng.normal(38, 9, size=n), 18, 65).astype(int)
    gender = rng.choice(["M", "F", "X"], size=n, p=[0.70, 0.295, 0.005])
    smoker_status = rng.choice(["NS", "S", "Unknown"], size=n, p=[0.78, 0.17, 0.05])
    bmi = np.clip(rng.normal(24.5, 3.5, size=n), 16.0, 45.0).round(1)

    # ---- Sum assured (lognormal, rounded to lakh) ----
    sa_log = rng.normal(np.log(5_000_000), 0.85, size=n)
    sum_assured = np.clip(np.exp(sa_log), 1_000_000, 50_000_000).astype(int)
    sum_assured = (sum_assured // 100_000 * 100_000).astype(int)

    # ---- Policy term & premium frequency ----
    terms = [5, 10, 15, 20, 25, 30, 40]
    term_p = [0.04, 0.10, 0.14, 0.28, 0.20, 0.18, 0.06]
    policy_term = rng.choice(terms, size=n, p=term_p)
    premium_frequency = rng.choice(
        ["Annual", "Semi-Annual", "Quarterly", "Monthly"],
        size=n, p=[0.45, 0.10, 0.10, 0.35],
    )

    # ---- Annualised premium: rough scaling with SA, age, term, smoker ----
    base_rate_per_lakh = 70 + 2.5 * (issue_age - 30)
    smoker_load = np.where(smoker_status == "S", 1.6, 1.0)
    term_factor = 0.9 + 0.02 * (policy_term - 20)
    annualised_premium = (base_rate_per_lakh * (sum_assured / 100_000)
                          * smoker_load * term_factor)
    annualised_premium = np.clip(annualised_premium * rng.uniform(0.85, 1.15, n),
                                 5_000, 200_000).round(0)

    # ---- Medical UW route — depends on SA and age ----
    def uw_route(sa, age):
        if sa < 1_500_000 and age < 45:
            return rng.choice(["NoMed", "TeleMed"], p=[0.7, 0.3])
        if sa < 5_000_000:
            return rng.choice(["TeleMed", "FullMed"], p=[0.6, 0.4])
        return "FullMed"

    medical_uw_route = np.array([uw_route(sa, a) for sa, a in zip(sum_assured, issue_age)])
    region = rng.choice(["Tier1", "Tier2", "Tier3"], size=n, p=[0.55, 0.30, 0.15])

    # ---- Lapse: piecewise-constant hazard, sampled by inverse-survival ----
    h1 = -np.log(0.84) / 13.0      # months 1-13:  16% lapsed by month 13
    h2 = -np.log(0.88) / 12.0      # months 14-25: 12% annualised on remaining
    h3 = 0.004                     # months 26+:   ~5%/year long-term

    u = rng.random(size=n)         # u = S(T), uniform on (0, 1)
    S13 = 0.84
    S25 = S13 * 0.88               # 0.7392
    lapse_time = np.empty(n)

    in_A = u > S13                 # lapsed in months 1-13
    if in_A.any():
        lapse_time[in_A] = -np.log(u[in_A]) / h1

    in_B = (u <= S13) & (u > S25)  # lapsed in months 14-25
    if in_B.any():
        lapse_time[in_B] = 13 - np.log(u[in_B] / S13) / h2

    in_C = u <= S25                # lapsed after month 25 (or not at all in obs)
    if in_C.any():
        lapse_time[in_C] = 25 - np.log(u[in_C] / S25) / h3

    term_months = policy_term * 12
    lapse_time = np.minimum(lapse_time, term_months)

    obs_date = pd.Timestamp("2026-05-15")  # workshop date = observation cut-off
    months_observed = ((obs_date.to_period("M") - issue_date.dt.to_period("M"))
                       .apply(lambda x: x.n)).values

    lapse_flag = ((lapse_time <= months_observed) & (lapse_time < term_months)).astype(int)
    lapse_duration_months = np.where(lapse_flag == 1, np.round(lapse_time).astype(int), -1)

    # ---- Mortality: q_x = 0.001 × 2^((age-30)/10), smokers × 2 ----
    base_q30 = 0.001
    smoker_mult = np.where(
        smoker_status == "S", 2.0,
        np.where(smoker_status == "NS", 1.0, 1.4),
    )

    inforce_months = np.minimum(np.minimum(lapse_time, term_months), months_observed)
    inforce_years_int = np.maximum(np.floor(inforce_months / 12.0).astype(int), 0)

    def cumulative_q(start_age, years_lived, mult):
        cum = 0.0
        for y in range(years_lived):
            age = start_age + y
            cum += base_q30 * (2 ** ((age - 30) / 10.0)) * mult
        return cum

    cum_qs = np.array([
        cumulative_q(a, y, m)
        for a, y, m in zip(issue_age, inforce_years_int, smoker_mult)
    ])
    claim_flag = (rng.random(size=n) < cum_qs).astype(int)

    # If death and lapse both flagged, death wins; revert the lapse.
    revert = (claim_flag == 1) & (lapse_flag == 1)
    lapse_flag[revert] = 0
    lapse_duration_months[revert] = -1

    return pd.DataFrame({
        "policy_id":              [f"ABC-LIF-{str(i + 1).zfill(7)}" for i in range(n)],
        "issue_date":             issue_date.dt.strftime("%Y-%m-%d"),
        "issue_age":              issue_age,
        "gender":                 gender,
        "sum_assured_inr":        sum_assured,
        "policy_term_years":      policy_term,
        "premium_frequency":      premium_frequency,
        "annualised_premium_inr": annualised_premium,
        "smoker_status":          smoker_status,
        "bmi":                    bmi,
        "medical_uw_route":       medical_uw_route,
        "region":                 region,
        "lapse_flag":             lapse_flag,
        "lapse_duration_months":  lapse_duration_months,
        "claim_flag":             claim_flag,
    })


life = build_life()
life.to_csv(DATA_DIR / "abc_life_term_cohort.csv", index=False)
print(f"Wrote {len(life):,} rows -> {DATA_DIR/'abc_life_term_cohort.csv'}")
life.head()


## 4. Verify headline figures

Each headline figure must land within ~5% of its locked target. Frequencies, severities, and pure premiums are computed using **aggregate** actuarial definitions (`Σ claims / Σ exposure`, `Σ amount / Σ count`) rather than mean-of-ratios.


In [ ]:
def pct_diff(target, observed):
    return abs(observed - target) / target * 100 if target else float("nan")


def check(label, target, observed, fmt="{:.2f}", unit="", tol=5.0):
    diff = pct_diff(target, observed)
    flag = "OK   " if diff <= tol else "WIDE "
    print(f"  [{flag}] {label:<28s} target {fmt.format(target)}{unit}"
          f"   observed {fmt.format(observed)}{unit}   ({diff:+.1f}%)")


print("=" * 72)
print("MOTOR — ABC General")
print("=" * 72)
agg_freq = motor["claim_count"].sum() / motor["exposure_years"].sum()
agg_sev = motor["claim_amount_inr"].sum() / motor["claim_count"].sum()
agg_pp = motor["claim_amount_inr"].sum() / motor["exposure_years"].sum()
fraud_rate = motor.loc[motor["claim_count"] > 0, "fraud_flag"].mean()

check("avg vehicle age (years)", 4.7, motor["vehicle_age_years"].mean(), "{:.2f}")
check("frequency (%)",            8.0, agg_freq * 100, "{:.2f}")
check("average severity (₹)",     38_000, agg_sev, "{:,.0f}", " ")
check("pure premium (₹)",         3_040, agg_pp, "{:,.0f}", " ")
check("fraud rate of claims (%)", 2.0, fraud_rate * 100, "{:.2f}")

print()
print("=" * 72)
print("HEALTH — ABC Health")
print("=" * 72)
n_pol = health["policy_id"].nunique()
h_freq = health["hospitalisation_count"].sum() / health["exposure_years"].sum()
h_sev = health["claim_amount_inr"].sum() / health["hospitalisation_count"].sum()
h_pp = health["claim_amount_inr"].sum() / health["exposure_years"].sum()

check("policies",                 3_200, n_pol, "{:,.0f}")
check("covered lives",            8_000, len(health), "{:,.0f}")
check("hosp frequency (%)",       4.2, h_freq * 100, "{:.2f}")
check("average severity (₹)",     62_000, h_sev, "{:,.0f}", " ")
check("pure premium (₹)",         2_604, h_pp, "{:,.0f}", " ")

print()
print("=" * 72)
print("LIFE — ABC Life Term Cohort")
print("=" * 72)
issue_dt = pd.to_datetime(life["issue_date"])
obs_date = pd.Timestamp("2026-05-15")
months_obs = ((obs_date.to_period("M") - issue_dt.dt.to_period("M"))
              .apply(lambda x: x.n)).astype(int)

eligible_13 = months_obs >= 13
lapsed_by_13 = (life["lapse_flag"] == 1) & (life["lapse_duration_months"] <= 13)
persistency_13 = 1 - (lapsed_by_13 & eligible_13).sum() / eligible_13.sum()

eligible_25 = months_obs >= 25
in_force_at_13 = ~lapsed_by_13
lapsed_yr2 = ((life["lapse_flag"] == 1)
              & (life["lapse_duration_months"] > 13)
              & (life["lapse_duration_months"] <= 25))
denom = (eligible_25 & in_force_at_13).sum()
yr2_lapse = (lapsed_yr2 & eligible_25 & in_force_at_13).sum() / denom

check("policies issued",          50_000, len(life), "{:,.0f}")
check("13-month persistency (%)", 84.0, persistency_13 * 100, "{:.1f}")
check("year-2 lapse (%)",         12.0, yr2_lapse * 100, "{:.1f}")

# Empirical q_x for issue-age 35 NS, blended across observed years
sub = life[(life["issue_age"] == 35) & (life["smoker_status"] == "NS")].copy()
sub_idx = sub.index
sub_months = months_obs[sub_idx]
sub_term_months = sub["policy_term_years"].values * 12
sub_lapse_dur = np.where(sub["lapse_flag"].values == 1,
                         sub["lapse_duration_months"].values, sub_term_months)
sub_inforce_months = np.minimum(sub_months, np.minimum(sub_lapse_dur, sub_term_months))
exp_yrs = (sub_inforce_months / 12).sum()
claims = sub["claim_flag"].sum()
q_blended = claims / exp_yrs if exp_yrs > 0 else float("nan")
formula_q35 = 0.001 * (2 ** 0.5)
print(f"  [INFO ] q_x age 35 NS year 1 (formula)         {formula_q35:.5f}")
print(f"  [INFO ] q_x age 35 NS observed (blended ages)  {q_blended:.5f}   (n={len(sub):,})")
print()
print("All headline checks complete.")


## Wrap-up

You should now have three CSVs in `data/`:

- `abc_motor_2024.csv` — 5,000 policies × 17 columns
- `abc_health_2024.csv` — ~7,700 lives × 14 columns (across 3,200 policies)
- `abc_life_term_cohort.csv` — 50,000 policies × 15 columns

**Where to next:** open `01_data_foundations.ipynb` for the Session 1 Part 2 walkthrough that loads and inspects these files.

**Companion slides:** none — this notebook is pre-workshop infrastructure, not a teaching segment.

**Reproducibility:** every run with `numpy.random.default_rng(42)` produces byte-identical CSVs. Change the seed only with intent.

This notebook demonstrated: deterministic generation of three hypothetical insurance datasets that hit their locked headline figures (frequency, severity, pure premium, persistency, lapse) within ~5%.
